# RescueRoute — benchmark thuật toán

Đo thời gian, peak Python heap, optimality và completeness theo [`docs/development/algorithm-benchmarking.md`](../docs/development/algorithm-benchmarking.md). Mở từ thư mục gốc RescueRoute bằng kernel `.venv`. Thuật toán hiện tại là Python thuần: GPU chỉ được ghi vào environment snapshot, không dùng để tính toán.

In [ ]:
# ── Đảm bảo dependencies có mặt trong bất kỳ kernel nào đang dùng ──────────
# Cell này an toàn để chạy nhiều lần; pip bỏ qua nếu đã cài đủ version.
%pip install --quiet pandas numpy matplotlib

In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import statistics
import subprocess
import sys
import time
import tracemalloc
from collections import defaultdict, deque
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Locate the repository root regardless of the CWD Jupyter happens to use.
# VS Code / Jupyter Lab often sets CWD to the *directory containing the
# notebook* (scripts/) rather than the project root.  Walk upward until we
# find a directory that looks like the RescueRoute repo root.
# ---------------------------------------------------------------------------
def _find_repo_root(start: Path, marker: str = 'backend') -> Path:
    """Walk up from *start* until a directory containing *marker/* is found."""
    candidate = start.resolve()
    for _ in range(10):  # safety limit: stop at filesystem root after 10 hops
        if (candidate / marker).is_dir():
            return candidate
        parent = candidate.parent
        if parent == candidate:  # reached filesystem root
            break
        candidate = parent
    raise RuntimeError(
        f"Không tìm thấy thư mục gốc RescueRoute (tìm '{marker}/' từ {start.resolve()}). "
        "Hãy chắc chắn notebook được mở trong workspace RescueRoute."
    )

ROOT = _find_repo_root(Path.cwd())
print(f'ROOT = {ROOT}')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from backend.app.algorithms.graph_search.astar.astar import solve_astar
from backend.app.algorithms.graph_search.bfs.bfs import solve_bfs
from backend.app.algorithms.graph_search.dfs.dfs import solve_dfs
from backend.app.algorithms.graph_search.dijkstra.dijkstra import solve_dijkstra
from backend.app.algorithms.graph_search.ucs.ucs import solve_ucs
from backend.app.algorithms.graph_search.trace_history import SearchFailure
from backend.app.algorithms.optimization.hill_climbing.algorithm import solve_hill_climbing
from backend.app.algorithms.optimization.genetic_algorithm.algorithm import solve_genetic_algorithm
from backend.app.algorithms.optimization.held_karp.algorithm import optimize_held_karp
from backend.app.algorithms.optimization.nearest_neighbor.algorithm import optimize_nearest_neighbor
from backend.app.algorithms.optimization.simulated_annealing.simulated_annealing import solve_simulated_annealing


## Cấu hình cố định

Không đổi các giá trị này trong cùng một bảng. `PIN_TO_ONE_LOGICAL_CPU` chỉ dành cho benchmark một-lõi; để `False` cho workload ứng dụng bình thường.

In [ ]:
TIME_REPEATS = 21
MEMORY_REPEATS = 7
WARMUP_RUNS = 3
RANDOM_SEED = 20260817
PIN_TO_ONE_LOGICAL_CPU = False
LOGICAL_CPU_INDEX = 0
WAYPOINT_COUNT = 6  # Held-Karp implementation giới hạn 10 waypoint.

EDGE_PATH = ROOT / 'data/samples/HCMUS_surrounding_filter/Minimap_ouput/edges.csv'
NODE_PATH = ROOT / 'data/samples/HCMUS_surrounding_filter/Minimap_ouput/nodes.csv'
ARTIFACT_DIR = ROOT / 'artifacts/benchmarks'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def command_output(command: list[str]) -> str | None:
    try:
        completed = subprocess.run(command, capture_output=True, text=True, check=False)
    except OSError:
        return None
    return completed.stdout.strip() or None


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def detect_gpu() -> str:
    nvidia = command_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'])
    if nvidia:
        return nvidia
    if platform.system() == 'Windows':
        command = "(Get-CimInstance Win32_VideoController | ForEach-Object { $_.Name }) -join '; '"
        return command_output(['powershell', '-NoProfile', '-Command', command]) or 'Không phát hiện GPU'
    return 'Không phát hiện GPU'


def total_memory_bytes() -> int | None:
    if platform.system() == 'Windows':
        raw = command_output(['powershell', '-NoProfile', '-Command', '(Get-CimInstance Win32_ComputerSystem).TotalPhysicalMemory'])
        return int(raw) if raw and raw.isdigit() else None
    try:
        return os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')
    except (AttributeError, ValueError, OSError):
        return None


def safe_package_version(name: str) -> str:
    """Return the installed version of *name*, or 'not installed' if absent."""
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return 'not installed'


def pin_to_one_logical_cpu(cpu_index: int) -> str:
    if cpu_index < 0 or cpu_index >= (os.cpu_count() or 1):
        raise ValueError(f'Logical CPU {cpu_index} không tồn tại.')
    if hasattr(os, 'sched_setaffinity'):
        os.sched_setaffinity(0, {cpu_index})
        return f'pinned to logical CPU {cpu_index}'
    if platform.system() == 'Windows':
        import ctypes
        mask = ctypes.c_size_t(1 << cpu_index)
        if ctypes.windll.kernel32.SetProcessAffinityMask(ctypes.windll.kernel32.GetCurrentProcess(), mask) == 0:
            raise OSError(ctypes.get_last_error(), 'SetProcessAffinityMask failed')
        return f'pinned to logical CPU {cpu_index}'
    raise RuntimeError('CPU affinity không được platform này hỗ trợ.')


affinity_note = 'not pinned (representative application workload)'
if PIN_TO_ONE_LOGICAL_CPU:
    affinity_note = pin_to_one_logical_cpu(LOGICAL_CPU_INDEX)
cpu_command = "(Get-CimInstance Win32_Processor | ForEach-Object { $_.Name.Trim() }) -join '; '"
environment = {
    'captured_at_utc': datetime.now(timezone.utc).isoformat(),
    'platform': platform.platform(), 'python': sys.version,
    'cpu_model': command_output(['powershell', '-NoProfile', '-Command', cpu_command]) if platform.system() == 'Windows' else platform.processor(),
    'logical_cpu_count': os.cpu_count(), 'physical_memory_bytes': total_memory_bytes(),
    'gpu_detected_for_reporting_only': detect_gpu(), 'gpu_used_by_benchmark': False,
    'cpu_affinity': affinity_note, 'git_commit': command_output(['git', 'rev-parse', 'HEAD']),
    'git_status_porcelain': command_output(['git', 'status', '--porcelain']),
    'dataset_sha256': {'edges.csv': file_sha256(EDGE_PATH), 'nodes.csv': file_sha256(NODE_PATH)},
    'packages': {name: safe_package_version(name) for name in ('pandas', 'numpy', 'matplotlib')},
    'benchmark_config': {'time_repeats': TIME_REPEATS, 'memory_repeats': MEMORY_REPEATS, 'warmup_runs': WARMUP_RUNS, 'random_seed': RANDOM_SEED, 'waypoint_count': WAYPOINT_COUNT, 'edge_cost': 'estimated_time_s'},
}
pd.Series(environment).to_frame('value')


In [ ]:
edges = pd.read_csv(EDGE_PATH)
nodes = pd.read_csv(NODE_PATH).rename(columns={'_id': 'node_id', 'long': 'longitude', 'lat': 'latitude'})
graph: dict[int, dict[int, dict[str, float]]] = defaultdict(dict)
for edge in edges.itertuples(index=False):
    source, target = int(edge.source_node_id), int(edge.target_node_id)
    graph[source][target] = {'total_cost': float(edge.estimated_time_s), 'distance': float(edge.distance_m), 'estimated_time': float(edge.estimated_time_s)}
    graph.setdefault(target, {})
graph = dict(graph)
coordinates = {int(row.node_id): (float(row.latitude), float(row.longitude)) for row in nodes.itertuples(index=False)}
max_speed_mps = float(edges.actual_speed_kph.max()) / 3.6

def largest_strongly_connected_component() -> list[int]:
    # Kosaraju: chọn SCC để mọi pairwise route của multi-stop đều tồn tại.
    reverse_graph: dict[int, list[int]] = {node_id: [] for node_id in graph}
    for source, neighbours in graph.items():
        for target in neighbours:
            reverse_graph[target].append(source)
    visited, finish_order = set(), []
    for root in sorted(graph):
        if root in visited:
            continue
        visited.add(root)
        stack = [(root, False)]
        while stack:
            node_id, exiting = stack.pop()
            if exiting:
                finish_order.append(node_id)
                continue
            stack.append((node_id, True))
            for neighbour in reversed(sorted(graph[node_id])):
                if neighbour not in visited:
                    visited.add(neighbour)
                    stack.append((neighbour, False))
    assigned, components = set(), []
    for root in reversed(finish_order):
        if root in assigned:
            continue
        assigned.add(root)
        component, stack = [], [root]
        while stack:
            node_id = stack.pop()
            component.append(node_id)
            for neighbour in sorted(reverse_graph[node_id]):
                if neighbour not in assigned:
                    assigned.add(neighbour)
                    stack.append(neighbour)
        components.append(sorted(component))
    return max(components, key=lambda component: (len(component), -component[0]))

strongly_connected_nodes = largest_strongly_connected_component()
if len(strongly_connected_nodes) < WAYPOINT_COUNT + 2:
    raise RuntimeError('SCC lớn nhất không đủ node cho workload benchmark.')
start_node = strongly_connected_nodes[len(strongly_connected_nodes) // 4]
goal_node = strongly_connected_nodes[(3 * len(strongly_connected_nodes)) // 4]
if start_node == goal_node:
    candidates = [n for n in strongly_connected_nodes if n != start_node]
    if candidates:
        goal_node = candidates[-1]

def haversine_meters(a: tuple[float, float], b: tuple[float, float]) -> float:
    lat1, lon1, lat2, lon2 = map(math.radians, (*a, *b))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    value = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 6_371_000 * 2 * math.asin(math.sqrt(value))
    
def travel_time_lower_bound(node_id: int, target_id: int) -> float:
    return haversine_meters(coordinates[node_id], coordinates[target_id]) / max_speed_mps

print(f'Graph: {len(graph):,} nodes, {len(edges):,} directed edges; largest SCC: {len(strongly_connected_nodes):,} nodes')
print(f'Graph-search workload: start={start_node}, goal={goal_node}, max speed={max_speed_mps:.3f} m/s')


In [ ]:
def percentile(values: list[float], fraction: float) -> float:
    ordered = sorted(values)
    index = (len(ordered) - 1) * fraction
    lower, upper = math.floor(index), math.ceil(index)
    return ordered[lower] if lower == upper else ordered[lower] + (ordered[upper] - ordered[lower]) * (index - lower)

def objective_cost(result: dict[str, Any] | None) -> float | None:
    if result is None:
        return None
    value = result.get('objective_cost', result.get('total_cost'))
    return None if value is None else float(value)

def invoke_safely(call: Callable[[], dict[str, Any]]) -> tuple[bool, dict[str, Any] | None, str | None]:
    try:
        return True, call(), None
    except SearchFailure as error:
        return False, error.result, str(error)
    except Exception as error:
        return False, None, f'{type(error).__name__}: {error}'

def outcome_signature(succeeded: bool, result: dict[str, Any] | None, error: str | None) -> tuple[Any, ...]:
    path = () if result is None else tuple(result.get('path', result.get('visiting_order', [])))
    return succeeded, path, objective_cost(result), error

def benchmark(name: str, call: Callable[[], dict[str, Any]], *, theory_optimality: str, theory_completeness: str, oracle_cost: float | None) -> dict[str, Any]:
    print(f'  [{name}] warm-up...', end=' ', flush=True)
    for _ in range(WARMUP_RUNS):
        invoke_safely(call)
    time_samples_ms, memory_samples_mib, outcomes = [], [], []
    previous_gc_state = gc.isenabled()
    gc.disable()
    print(f'timing ({TIME_REPEATS}x)...', end=' ', flush=True)
    try:
        for _ in range(TIME_REPEATS):
            started = time.perf_counter_ns()
            succeeded, result, error = invoke_safely(call)
            time_samples_ms.append((time.perf_counter_ns() - started) / 1_000_000)
            outcomes.append((succeeded, result, error))
    finally:
        if previous_gc_state:
            gc.enable()
    print(f'memory ({MEMORY_REPEATS}x)...', end=' ', flush=True)
    for _ in range(MEMORY_REPEATS):
        gc.collect()
        tracemalloc.start()
        baseline_current, _ = tracemalloc.get_traced_memory()
        invoke_safely(call)
        _current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        memory_samples_mib.append(max(0, peak - baseline_current) / (1024 * 1024))
    succeeded, reference_result, reference_error = outcomes[0]
    measured_cost = objective_cost(reference_result) if succeeded else None
    gap = None if oracle_cost in (None, 0) or measured_cost is None else (measured_cost - oracle_cost) * 100 / oracle_cost
    median_ms = statistics.median(time_samples_ms)
    print(f'done. median={median_ms:.2f} ms')
    return {
        'algorithm': name, 'runtime_median_ms': median_ms, 'runtime_p95_ms': percentile(time_samples_ms, 0.95),
        'peak_python_heap_median_mib': statistics.median(memory_samples_mib), 'peak_python_heap_max_mib': max(memory_samples_mib),
        'objective_cost': measured_cost, 'oracle_cost': oracle_cost, 'cost_gap_to_oracle_pct': gap,
        'completed_all_runs': all(item[0] for item in outcomes),
        'deterministic_across_timed_runs': len({outcome_signature(*item) for item in outcomes}) == 1,
        'first_failure': reference_error, 'theoretical_optimality': theory_optimality, 'theoretical_completeness': theory_completeness,
        'time_repeats': TIME_REPEATS, 'memory_repeats': MEMORY_REPEATS,
    }


## Benchmark tìm đường hai điểm

Dijkstra là oracle vì `estimated_time_s` không âm. A* dùng Haversine / tốc độ lớn nhất quan sát được, một lower bound của travel time theo cost profile này. Hill Climbing được giữ trong bảng kể cả khi nó dừng ở local optimum; khi đó `completed_all_runs=False` là kết quả cần báo cáo, không phải lỗi benchmark.

In [ ]:
oracle = solve_dijkstra(graph, start_node, goal_node)
oracle_cost = float(oracle['total_cost'])
graph_search_cases = [
    ('BFS', lambda: solve_bfs(graph, start_node, goal_node), 'minimum hops only; not minimum travel time', 'complete on a finite graph'),
    ('DFS', lambda: solve_dfs(graph, start_node, goal_node), 'no travel-time optimality guarantee', 'complete on a finite graph for solve_dfs; bounded variants are not'),
    ('UCS', lambda: solve_ucs(graph, start_node, goal_node), 'minimum travel time when costs are non-negative', 'complete on a finite graph with non-negative costs'),
    ('Dijkstra', lambda: solve_dijkstra(graph, start_node, goal_node), 'minimum travel time when costs are non-negative', 'complete on a finite graph with non-negative costs'),
    ('A*', lambda: solve_astar(graph, start_node, goal_node, heuristic=travel_time_lower_bound, heuristic_is_admissible=True), 'minimum travel time if the lower-bound heuristic is admissible', 'complete on a finite graph with non-negative costs and finite heuristic'),
    ('Hill Climbing', lambda: solve_hill_climbing(graph, start_node, goal_node, travel_time_lower_bound), 'no global travel-time optimality guarantee', 'not complete; it can stop at a local optimum or dead end'),
]
print('=== Graph-search benchmark ===')
graph_search_rows = []
for name, call, optimality, completeness in graph_search_cases:
    graph_search_rows.append(
        benchmark(name, call, theory_optimality=optimality, theory_completeness=completeness, oracle_cost=oracle_cost)
    )
graph_search_table = pd.DataFrame(graph_search_rows).sort_values('runtime_median_ms')
graph_search_table


## Benchmark tối ưu nhiều điểm dừng

Pairwise matrix được tạo trước khi đo bằng Dijkstra; setup này không nằm trong timing/memory của thuật toán tối ưu. Held-Karp là oracle cho thứ tự waypoint.

In [ ]:
locations = [start_node, *strongly_connected_nodes[1:WAYPOINT_COUNT + 1], goal_node]
locations = list(dict.fromkeys(locations))
if len(locations) != WAYPOINT_COUNT + 2:
    raise RuntimeError('Không đủ node khác nhau để tạo workload nhiều điểm dừng.')
multi_start, *middle, multi_goal = locations
distance_matrix: dict[int, dict[int, float]] = {node_id: {} for node_id in locations}
print(f'Xây dựng pairwise Dijkstra matrix ({len(locations)}x{len(locations)})...', flush=True)
for source in locations:
    for target in locations:
        distance_matrix[source][target] = 0.0 if source == target else float(solve_dijkstra(graph, source, target)['total_cost'])
pair_costs = {(source, target): cost for source, row in distance_matrix.items() for target, cost in row.items()}
held_karp_oracle = optimize_held_karp(multi_start, middle, multi_goal, pair_costs)
multi_oracle_cost = float(held_karp_oracle['objective_cost'])
multi_stop_cases = [
    ('Nearest Neighbor', lambda: optimize_nearest_neighbor(multi_start, middle, multi_goal, pair_costs), 'approximate except <= 1 waypoint', 'not complete on sparse matrices because a greedy choice can block the final route'),
    ('Held-Karp', lambda: optimize_held_karp(multi_start, middle, multi_goal, pair_costs), 'globally optimal waypoint order over this pairwise matrix', 'complete for <= 10 waypoint and a feasible tour'),
    ('Simulated Annealing', lambda: solve_simulated_annealing(locations, distance_matrix, multi_start, multi_goal, seed=RANDOM_SEED), 'approximate; no global guarantee', 'returns a feasible order for a complete finite matrix, not a general completeness guarantee'),
    ('Genetic Algorithm', lambda: solve_genetic_algorithm(locations, distance_matrix, multi_start, multi_goal, population_size=20, generations=30, seed=RANDOM_SEED), 'approximate; no global guarantee', 'returns a feasible order for a complete finite matrix, not a general completeness guarantee'),
]
print('=== Multi-stop benchmark ===')
multi_stop_rows = []
for name, call, optimality, completeness in multi_stop_cases:
    multi_stop_rows.append(
        benchmark(name, call, theory_optimality=optimality, theory_completeness=completeness, oracle_cost=multi_oracle_cost)
    )
multi_stop_table = pd.DataFrame(multi_stop_rows).sort_values('runtime_median_ms')
multi_stop_table


## Tóm tắt kết quả

So sánh nhanh median/p95 và cost gap giữa các thuật toán trong cùng nhóm. Không so sánh thời gian giữa hai nhóm vì không gian bài toán khác nhau.

In [ ]:
_DISPLAY_COLS = ['algorithm', 'runtime_median_ms', 'runtime_p95_ms',
                 'peak_python_heap_median_mib', 'cost_gap_to_oracle_pct',
                 'completed_all_runs', 'deterministic_across_timed_runs']

print('=== Graph-search summary ===')
print(graph_search_table[_DISPLAY_COLS].to_string(index=False))

print()
print('=== Multi-stop summary ===')
print(multi_stop_table[_DISPLAY_COLS].to_string(index=False))

# Fastest and closest-to-optimal per group
gs_fastest = graph_search_table.iloc[0]
gs_optimal = graph_search_table[graph_search_table['cost_gap_to_oracle_pct'].notna()].sort_values('cost_gap_to_oracle_pct').iloc[0]
ms_fastest = multi_stop_table.iloc[0]
ms_best_gap = multi_stop_table[multi_stop_table['cost_gap_to_oracle_pct'].notna()].sort_values('cost_gap_to_oracle_pct').iloc[0]

print()
print('--- Quick conclusions ---')
print(f'Graph-search fastest  : {gs_fastest["algorithm"]} ({gs_fastest["runtime_median_ms"]:.2f} ms median)')
print(f'Graph-search best gap : {gs_optimal["algorithm"]} ({gs_optimal["cost_gap_to_oracle_pct"]:.2f}% vs Dijkstra oracle)')
print(f'Multi-stop fastest    : {ms_fastest["algorithm"]} ({ms_fastest["runtime_median_ms"]:.2f} ms median)')
print(f'Multi-stop best gap   : {ms_best_gap["algorithm"]} ({ms_best_gap["cost_gap_to_oracle_pct"]:.2f}% vs Held-Karp oracle)')
print()
print('NOTE: so sánh thời gian chỉ hợp lệ trong cùng nhóm và cùng environment snapshot.')


## Trực quan hóa kết quả (Plots)

Vẽ biểu đồ so sánh thời gian (median & p95), bộ nhớ (peak Python heap) và độ lệch chi phí (cost gap) cho cả hai nhóm thuật toán.

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# ---------------------------------------------------------------------------
# Figure 1: Graph-search benchmark
# ---------------------------------------------------------------------------
fig_gs, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig_gs.suptitle('RescueRoute — Graph Search Benchmark (Tìm đường 2 điểm)', fontsize=13, fontweight='bold')

df_gs_plot = graph_search_table.sort_values('runtime_median_ms')
y_gs = np.arange(len(df_gs_plot))
colors_gs = ['#3498DB', '#2ECC71', '#1ABC9C', '#F39C12', '#E67E22', '#E74C3C']

bars1 = ax1.barh(y_gs, df_gs_plot['runtime_median_ms'], color=colors_gs[:len(df_gs_plot)], alpha=0.85, edgecolor='black', linewidth=0.7)
for i, r in df_gs_plot.reset_index().iterrows():
    ax1.text(r['runtime_median_ms'] + 0.2, i, f"{r['runtime_median_ms']:.2f} ms (p95: {r['runtime_p95_ms']:.2f})", va='center', fontsize=8.5, fontweight='bold')
ax1.set_yticks(y_gs)
ax1.set_yticklabels(df_gs_plot['algorithm'], fontsize=9.5, fontweight='bold')
ax1.set_xlabel('Median Wall-clock Time (ms)')
ax1.set_title('Thời gian thực thi (ms)', fontsize=11, fontweight='bold')
ax1.set_xlim(0, max(df_gs_plot['runtime_p95_ms']) * 1.25)
ax1.grid(True, linestyle='--', alpha=0.5)

bars2 = ax2.barh(y_gs, df_gs_plot['peak_python_heap_median_mib'], color='#2C3E50', alpha=0.85, edgecolor='black', linewidth=0.7)
for i, r in df_gs_plot.reset_index().iterrows():
    ax2.text(r['peak_python_heap_median_mib'] + 0.03, i, f"{r['peak_python_heap_median_mib']:.3f} MiB", va='center', fontsize=8.5)
ax2.set_yticks(y_gs)
ax2.set_yticklabels(df_gs_plot['algorithm'], fontsize=9.5)
ax2.set_xlabel('Peak Python Heap (MiB)')
ax2.set_title('Bộ nhớ đỉnh Python Heap (MiB)', fontsize=11, fontweight='bold')
ax2.set_xlim(0, max(df_gs_plot['peak_python_heap_median_mib']) * 1.25)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plot_gs_file = ARTIFACT_DIR / 'benchmark_graph_search.png'
plt.savefig(plot_gs_file, dpi=200, bbox_inches='tight')
plt.show()
print(f'Đã lưu biểu đồ graph-search: {plot_gs_file.relative_to(ROOT)}')

# ---------------------------------------------------------------------------
# Figure 2: Multi-stop optimization benchmark
# ---------------------------------------------------------------------------
fig_ms, (ax3, ax4) = plt.subplots(1, 2, figsize=(14, 5))
fig_ms.suptitle(f'RescueRoute — Multi-Stop Optimization Benchmark ({WAYPOINT_COUNT} Waypoints)', fontsize=13, fontweight='bold')

df_ms_plot = multi_stop_table.sort_values('runtime_median_ms')
y_ms = np.arange(len(df_ms_plot))
colors_ms = ['#27AE60', '#2980B9', '#D35400', '#8E44AD']

bars3 = ax3.barh(y_ms, df_ms_plot['runtime_median_ms'], color=colors_ms[:len(df_ms_plot)], alpha=0.85, edgecolor='black', linewidth=0.7)
for i, r in df_ms_plot.reset_index().iterrows():
    ax3.text(r['runtime_median_ms'] + 0.3, i, f"{r['runtime_median_ms']:.2f} ms (p95: {r['runtime_p95_ms']:.2f})", va='center', fontsize=8.5, fontweight='bold')
ax3.set_yticks(y_ms)
ax3.set_yticklabels(df_ms_plot['algorithm'], fontsize=9.5, fontweight='bold')
ax3.set_xlabel('Median Wall-clock Time (ms)')
ax3.set_title('Thời gian thực thi (ms)', fontsize=11, fontweight='bold')
ax3.set_xlim(0, max(df_ms_plot['runtime_p95_ms']) * 1.25)
ax3.grid(True, linestyle='--', alpha=0.5)

gaps = df_ms_plot['cost_gap_to_oracle_pct'].fillna(0.0)
gap_colors = ['#27AE60' if g == 0 else '#C0392B' for g in gaps]
bars4 = ax4.barh(y_ms, gaps, color=gap_colors, alpha=0.85, edgecolor='black', linewidth=0.7)
for i, r in df_ms_plot.reset_index().iterrows():
    gap = r['cost_gap_to_oracle_pct']
    cost = r['objective_cost']
    txt = f"+{gap:.2f}% ({cost:.1f}s)" if pd.notna(gap) and gap > 0 else f"0.0% (Oracle: {cost:.1f}s)"
    ax4.text(max(0, gap) + 0.2, i, txt, va='center', fontsize=8.5, fontweight='bold')
ax4.set_yticks(y_ms)
ax4.set_yticklabels(df_ms_plot['algorithm'], fontsize=9.5)
ax4.set_xlabel('Cost Gap vs Oracle (%)')
ax4.set_title('Độ lệch chi phí so với Oracle (%)', fontsize=11, fontweight='bold')
ax4.set_xlim(0, max(gaps) * 1.45 if max(gaps) > 0 else 10)
ax4.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plot_ms_file = ARTIFACT_DIR / 'benchmark_multi_stop.png'
plt.savefig(plot_ms_file, dpi=200, bbox_inches='tight')
plt.show()
print(f'Đã lưu biểu đồ multi-stop: {plot_ms_file.relative_to(ROOT)}')

# ---------------------------------------------------------------------------
# Figure 3: Combined 4-panel summary
# ---------------------------------------------------------------------------
fig_all, ((p1, p2), (p3, p4)) = plt.subplots(2, 2, figsize=(15, 9))
fig_all.suptitle('RescueRoute — Tổng hợp Benchmark Thuật toán', fontsize=14, fontweight='bold')

p1.barh(y_gs, df_gs_plot['runtime_median_ms'], color=colors_gs[:len(df_gs_plot)], alpha=0.85, edgecolor='black', linewidth=0.7)
for i, r in df_gs_plot.reset_index().iterrows():
    p1.text(r['runtime_median_ms'] + 0.2, i, f"{r['runtime_median_ms']:.2f} ms", va='center', fontsize=8, fontweight='bold')
p1.set_yticks(y_gs)
p1.set_yticklabels(df_gs_plot['algorithm'], fontsize=9)
p1.set_title('1. Tìm đường: Thời gian (ms)', fontsize=10.5, fontweight='bold')
p1.set_xlim(0, max(df_gs_plot['runtime_p95_ms']) * 1.2)
p1.grid(True, linestyle='--', alpha=0.5)

p2.barh(y_gs, df_gs_plot['peak_python_heap_median_mib'], color='#2C3E50', alpha=0.85, edgecolor='black', linewidth=0.7)
for i, r in df_gs_plot.reset_index().iterrows():
    p2.text(r['peak_python_heap_median_mib'] + 0.03, i, f"{r['peak_python_heap_median_mib']:.3f}M", va='center', fontsize=8)
p2.set_yticks(y_gs)
p2.set_yticklabels(df_gs_plot['algorithm'], fontsize=9)
p2.set_title('2. Tìm đường: Memory (MiB)', fontsize=10.5, fontweight='bold')
p2.set_xlim(0, max(df_gs_plot['peak_python_heap_median_mib']) * 1.25)
p2.grid(True, linestyle='--', alpha=0.5)

p3.barh(y_ms, df_ms_plot['runtime_median_ms'], color=colors_ms[:len(df_ms_plot)], alpha=0.85, edgecolor='black', linewidth=0.7)
for i, r in df_ms_plot.reset_index().iterrows():
    p3.text(r['runtime_median_ms'] + 0.3, i, f"{r['runtime_median_ms']:.2f} ms", va='center', fontsize=8, fontweight='bold')
p3.set_yticks(y_ms)
p3.set_yticklabels(df_ms_plot['algorithm'], fontsize=9)
p3.set_title('3. Nhiều điểm dừng: Thời gian (ms)', fontsize=10.5, fontweight='bold')
p3.set_xlim(0, max(df_ms_plot['runtime_p95_ms']) * 1.2)
p3.grid(True, linestyle='--', alpha=0.5)

p4.barh(y_ms, gaps, color=gap_colors, alpha=0.85, edgecolor='black', linewidth=0.7)
for i, r in df_ms_plot.reset_index().iterrows():
    gap = r['cost_gap_to_oracle_pct']
    txt = f"+{gap:.2f}%" if pd.notna(gap) and gap > 0 else "0.0%"
    p4.text(max(0, gap) + 0.2, i, txt, va='center', fontsize=8, fontweight='bold')
p4.set_yticks(y_ms)
p4.set_yticklabels(df_ms_plot['algorithm'], fontsize=9)
p4.set_title('4. Nhiều điểm dừng: Cost Gap (%)', fontsize=10.5, fontweight='bold')
p4.set_xlim(0, max(gaps) * 1.45 if max(gaps) > 0 else 10)
p4.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plot_all_file = ARTIFACT_DIR / 'benchmark_summary.png'
plt.savefig(plot_all_file, dpi=200, bbox_inches='tight')
plt.show()
print(f'Đã lưu biểu đồ tổng hợp: {plot_all_file.relative_to(ROOT)}')


In [ ]:
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
graph_csv = ARTIFACT_DIR / f'graph-search-{run_id}.csv'
multi_csv = ARTIFACT_DIR / f'multi-stop-{run_id}.csv'
environment_json = ARTIFACT_DIR / f'environment-{run_id}.json'
graph_search_table.to_csv(graph_csv, index=False)
multi_stop_table.to_csv(multi_csv, index=False)
environment.update({
    'run_id': run_id,
    'graph_search_workload': {'start_node': start_node, 'goal_node': goal_node, 'oracle_cost': oracle_cost},
    'multi_stop_workload': {'locations': locations, 'oracle_cost': multi_oracle_cost},
    'result_files': [
        str(graph_csv.relative_to(ROOT)),
        str(multi_csv.relative_to(ROOT)),
        str(plot_gs_file.relative_to(ROOT)),
        str(plot_ms_file.relative_to(ROOT)),
        str(plot_all_file.relative_to(ROOT)),
    ],
})
environment_json.write_text(json.dumps(environment, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu các file kết quả và biểu đồ:')
print('  • CSV graph-search:', graph_csv.relative_to(ROOT))
print('  • CSV multi-stop  :', multi_csv.relative_to(ROOT))
print('  • JSON environment:', environment_json.relative_to(ROOT))
print('  • PNG graph-search:', plot_gs_file.relative_to(ROOT))
print('  • PNG multi-stop  :', plot_ms_file.relative_to(ROOT))
print('  • PNG summary     :', plot_all_file.relative_to(ROOT))


## Ghi trong báo cáo

Ghi dataset hash, Git commit, CPU model, logical CPU count, RAM, GPU phát hiện (kèm câu `GPU not used`), power mode, CPU-affinity, số repeats, seed và các tên file output. Chỉ kết luận nhanh/chậm từ median/p95 của **cùng workload và cùng environment snapshot**. `completed_all_runs` không phải bằng chứng của completeness; dùng cột theoretical completeness và điều kiện kèm theo.